# Chapter 26 — Evaluating LLMs

Every number in Chapter 26, recomputed from `tfs/evals.py`. Each cell **asserts** what it checks,
so `python tests/run_notebooks.py 26` fails if the chapter and the code ever disagree.

In [ ]:
import math, sys, os, random
sys.path.insert(0, os.path.abspath(".."))      # run from notebooks/
sys.path.insert(0, os.path.abspath("."))       # or from the repo root
from tfs import evals as E

## 26.3 · Contamination signals
n-gram overlap needs the training corpus; Min-K% Prob needs only per-token log-probs.

In [ ]:
corpus = "the quick brown fox jumps over the lazy dog and then the fox runs into the forest to find food".split()
cg = E.ngrams(corpus, 5)
a = E.ngram_overlap("a quick brown fox jumps over the lazy dog today", cg, 5)
b = E.ngram_overlap("the fox jumps over the lazy cat and runs away", cg, 5)
print(round(a, 3), round(b, 3))
assert round(a, 3) == 0.667 and round(b, 3) == 0.167

seen   = [-0.2, -0.5, -0.1, -1.1, -0.3, -0.4, -0.9, -0.2, -0.6, -0.3]
unseen = [-0.4, -3.9, -0.2, -2.7, -0.6, -5.1, -0.8, -0.3, -1.9, -0.5]
means = sum(seen) / 10, sum(unseen) / 10
mink = E.min_k_prob(seen, 0.2), E.min_k_prob(unseen, 0.2)
print("means", means, "min-20%", mink)
assert round(means[0], 2) == -0.46 and round(means[1], 2) == -1.64
assert mink == (-1.0, -4.5)
assert round(means[0] - means[1], 2) == 1.18 and mink[0] - mink[1] == 3.5

## 26.4 · Scoring one answer

In [ ]:
print(E.exact_match("The Eiffel Tower.", "Eiffel Tower"), E.exact_match("The Eiffel Tower.", "Eiffel Tower", normalize=False))
print(E.normalize_answer("the tower in Paris"), E.token_f1("the tower in Paris", "Eiffel Tower"))
assert E.exact_match("The Eiffel Tower.", "Eiffel Tower") == 1
assert E.exact_match("The Eiffel Tower.", "Eiffel Tower", normalize=False) == 0
assert abs(E.token_f1("the tower in Paris", "Eiffel Tower") - 0.4) < 1e-12
assert E.majority_vote(["42", "41", "42.", "forty-two", "42"]) == "42"

## 26.5 · The resolution of a benchmark

In [ ]:
rows = [("AIME", 30, 0.50), ("HumanEval", 164, 0.90), ("GPQA Diamond", 198, 0.70),
        ("SWE-bench Verified", 500, 0.60), ("MMLU", 14042, 0.85)]
expect = {"AIME": (0.0913, 17.9, 36.2), "HumanEval": (0.0234, 4.6, 9.3), "GPQA Diamond": (0.0326, 6.4, 12.9),
          "SWE-bench Verified": (0.0219, 4.3, 8.7), "MMLU": (0.0030, 0.59, 1.2)}
for name, n, p in rows:
    se = E.standard_error(p, n)
    half = 100 * 1.959964 * se
    mde = 100 * E.mde_unpaired(p, n)
    print(f"{name:20s} n={n:6d}  SE={se:.4f}  +-{half:.2f} pts  MDE={mde:.1f} pts")
    s, h, m = expect[name]
    assert round(se, 4) == s and round(mde, 1) == m
    assert round(half, 2 if name == "MMLU" else 1) == h
lo, hi = E.ci95(0.70, 198)
assert (round(lo, 3), round(hi, 3)) == (0.636, 0.764)
assert round(100 / 30, 2) == 3.33

### Paired versus unpaired: the same 3-point gain
A right on 350 of 500; B right on 340 of those and 25 of A's failures.

In [ ]:
A = [1] * 350 + [0] * 150
B = [1] * 340 + [0] * 10 + [1] * 25 + [0] * 125
d = [y - x for x, y in zip(A, B)]
mean_d = sum(d) / 500
sd = (sum((x - mean_d) ** 2 for x in d) / 499) ** 0.5
se_paired = sd / 500 ** 0.5
se_unpaired = (0.7 * 0.3 / 500 + 0.73 * 0.27 / 500) ** 0.5
print(sum(B) / 500, d.count(1), d.count(-1), round(se_paired, 4), round(se_unpaired, 4), round(se_unpaired / se_paired, 2))
assert sum(B) == 365 and d.count(1) == 25 and d.count(-1) == 10
assert round(se_paired, 4) == 0.0118 and round(se_unpaired, 4) == 0.0285 and round(se_unpaired / se_paired, 1) == 2.4
pu = E.unpaired_bootstrap_diff(A, B); pp = E.paired_bootstrap_diff(A, B)
print("unpaired", pu, "paired", pp)
assert (round(pu[0], 3), round(pu[1], 3)) == (-0.024, 0.086)
assert (round(pp[0], 3), round(pp[1], 3)) == (0.008, 0.054)

### Items needed to see 70% -> 72%

In [ ]:
z = E.Z(0.975) + E.Z(0.8)
print(round(z, 3), E.n_items_unpaired(0.70, 0.72), E.n_items_paired(0.10, 0.02))
assert round(z, 3) == 2.802
assert E.n_items_unpaired(0.70, 0.72) == 8077
assert E.n_items_paired(0.10, 0.02) == 1955
assert round(E.n_items_unpaired(0.70, 0.72) / E.n_items_paired(0.10, 0.02), 1) == 4.1

## 26.6 · pass@k, and why the naive estimate is biased low

In [ ]:
print(E.pass_at_k(10, 3, 1), E.pass_at_k(10, 3, 5), E.naive_pass_at_k(10, 3, 5))
assert abs(E.pass_at_k(10, 3, 1) - 0.3) < 1e-12
assert math.comb(7, 5) == 21 and math.comb(10, 5) == 252
assert round(E.pass_at_k(10, 3, 5), 4) == 0.9167 and round(E.naive_pass_at_k(10, 3, 5), 4) == 0.8319
exact_u = E.expected_estimator(E.pass_at_k, 10, 0.3, 5)
exact_n = E.expected_estimator(E.naive_pass_at_k, 10, 0.3, 5)
print("E[unbiased]", exact_u, "E[naive]", exact_n, "truth", 1 - 0.7 ** 5)
assert abs(exact_u - (1 - 0.7 ** 5)) < 1e-12 and round(exact_n, 3) == 0.760 and round(1 - 0.7 ** 5, 3) == 0.832
# the product form agrees with the binomial form even where C(n, k) is enormous
assert abs(E.pass_at_k(200, 37, 50) - (1 - math.comb(163, 50) / math.comb(200, 50))) < 1e-12

## 26.7 · Judges

In [ ]:
rng = random.Random(0)
lm, lb, wins = [], [], []
for _ in range(805):
    b = rng.randint(150, 450); lb.append(b); lm.append(max(20, int(b + rng.gauss(120, 150))))
sd = (sum((a - b) ** 2 for a, b in zip(lm, lb)) / 805) ** 0.5
for a, b in zip(lm, lb):          # this judge rewards length and nothing else
    wins.append(int(rng.random() < 1 / (1 + math.exp(-1.5 * math.tanh((a - b) / sd)))))
res = E.length_controlled_winrate(wins, lm, lb)
print(sum(wins), round(sum(lm) / 805), round(sum(lb) / 805), {k: round(v, 3) for k, v in res.items()})
assert sum(wins) == 499 and round(sum(lm) / 805) == 417 and round(sum(lb) / 805) == 297
assert round(res["raw"], 3) == 0.620 and round(res["lc"], 3) == 0.474
assert round(res["phi"], 2) == 1.54 and round(res["theta"], 3) == -0.104

for v in [("first", "second"), ("first", "first"), ("second", "first"), ("second", "second")]:
    print(v, E.debias_pairwise(*v))
print([round(E.elo_win_prob(x), 3) for x in (30, 100, 400)])
assert [round(E.elo_win_prob(x), 3) for x in (30, 100, 400)] == [0.543, 0.640, 0.909]

## 26.8 · Agents: pass^k

In [ ]:
print(round(0.6 ** 8, 4), E.pass_hat_k(10, 8, 4), round(0.8 ** 4, 3))
assert round(0.6 ** 8, 4) == 0.0168
assert math.comb(8, 4) == 70 and math.comb(10, 4) == 210 and abs(E.pass_hat_k(10, 8, 4) - 1 / 3) < 1e-12
assert round(0.8 ** 4, 3) == 0.410
assert abs(E.expected_estimator(E.pass_hat_k, 10, 0.8, 4) - 0.8 ** 4) < 1e-12

## 26.9 · Cohen's kappa on 20 labels

In [ ]:
a, b = list("YYYYYYYYYYYYYYNNNNNN"), list("YYYYYYYYYYYYNNYYNNNN")
po = sum(x == y for x, y in zip(a, b)) / 20
pe = 0.7 * 0.7 + 0.3 * 0.3
print(po, pe, E.cohens_kappa(a, b))
assert po == 0.8 and abs(pe - 0.58) < 1e-12 and round(E.cohens_kappa(a, b), 3) == 0.524

## Exercise 26.3 · the cost of one AIME evaluation

In [ ]:
tokens = 30 * 64 * 15_000
print(tokens, tokens / 1e6 * 10, tokens / 2000 / 3600, round(100 * 1.959964 * E.standard_error(0.5, 30), 1))
assert tokens == 28_800_000 and tokens / 1e6 * 10 == 288.0 and tokens / 2000 / 3600 == 4.0
assert round(E.standard_error(0.5, 30), 3) == 0.091